# 서울시 100m GRID별 성별, 연령대(10대 ~ 70세 이상) 데이터 테이블 생성
- 목적: Huffs 문화시설 선호 확률을 반영한 3SFCA 접근성 모델의 설계 위해 격자별 인구 통계 데이터를 생성하는 노트북
- 방법: 기존 격자 데이터 테이블의 문화누리대상자 값에, 격자별 인구 통계 데이터(성별, 연령별 총 인구)의 비율 곱하여 카테고리별 인구수 추정

### 분석 순서: 데이터 불러오기 - EDA - 데이터 전처리 - 데이터 병합

In [28]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pathlib
import numpy as np
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap
import mapclassify as mc

salmon_cmap = LinearSegmentedColormap.from_list(
    "salmon_cmap",
    ["#fff5f0", "#fddbc7", "#f4a582", "#ef8a62", "#d6604d", "#b2182b"]
)

plt.rcParams["font.family"] = "Noto Sans KR"

BASE_PATH = pathlib.Path().resolve()

if BASE_PATH.name == "notebooks":
    BASE_PATH = BASE_PATH.parent
elif BASE_PATH.name != "analysis_table" and (BASE_PATH / "analysis_table").exists():
    BASE_PATH = BASE_PATH / "analysis_table"

INPUT_PATH = BASE_PATH / "data" / "input"
OUTPUT_PATH = BASE_PATH / "data" / "output"

DATA_PATH = INPUT_PATH
RAW_DATA = DATA_PATH / "raw"
PRO_DATA = OUTPUT_PATH

IMAGE_PATH = BASE_PATH / "image"
IMAGE_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
MAKING_TABLE_PATH = OUTPUT_PATH
WELFARE_PATH = DATA_PATH / "welfare"
WELFARE_PATH.mkdir(parents=True, exist_ok = True)

PROJECT_PATH = BASE_PATH.parent
GRID_PATH = PROJECT_PATH / "data" / "grid"
AGE_GENDER_GRID_PATH = GRID_PATH / "격자100m_성연령별인구_2024_10"

## 데이터 불러오기
- 격자 추정 인구 수
- 서울시 시군구_행정동 경계
- 격자별 인구 통계


In [ ]:
grid = gpd.read_file(OUTPUT_PATH / "서울시_100m_문화누리추정인구수.gpkg")
hjd = gpd.read_file(OUTPUT_PATH / "서울시_시군구_행정동_경계.gpkg")


print(f'\ngrid 칼럼: {grid.columns}')
print(f'\n행정동 칼럼: {hjd.columns}')
display(grid.head(10))
display(hjd.head(10))


grid 칼럼: Index(['GRID_CD', '행정동코드', '시군구', '행정동', '중심점_x', '중심점_y', '원본_인구수', '주거면적',
       '주택수', '공시지가', '추정_인구수', '문화누리대상자_추정_인구수', 'GRID_CD_500', 'geometry'],
      dtype='str')

행정동 칼럼: Index(['ADM_CD', '시군구', '행정동', 'geometry'], dtype='str')


,GRID_CD,행정동코드,시군구,행정동,중심점_x,중심점_y,원본_인구수,주거면적,주택수,공시지가,추정_인구수,문화누리대상자_추정_인구수,GRID_CD_500,geometry
0,다사603367,11220670,서초구,양재2동,960350.0,1936750.0,0.0,0.0,0.0,21500.000000,0,0,다사60a36b,"POLYGON ((960300 1936700, 960300 1936800, 9604..."
1,다사604367,11220670,서초구,양재2동,960450.0,1936750.0,0.0,0.0,0.0,21949.995000,0,0,다사60a36b,"POLYGON ((960400 1936700, 960400 1936800, 9605..."
2,다사605367,11220670,서초구,양재2동,960550.0,1936750.0,0.0,0.0,0.0,30233.336667,0,0,다사60b36b,"POLYGON ((960500 1936700, 960500 1936800, 9606..."
3,다사615367,11220680,서초구,내곡동,961550.0,1936750.0,0.0,0.0,0.0,77118.103333,0,0,다사61b36b,"POLYGON ((961500 1936700, 961500 1936800, 9616..."
4,다사602368,11220670,서초구,양재2동,960250.0,1936850.0,0.0,0.0,0.0,21699.990000,0,0,다사60a36b,"POLYGON ((960200 1936800, 960200 1936900, 9603..."
5,다사603368,11220670,서초구,양재2동,960350.0,1936850.0,0.0,0.0,0.0,21949.995000,0,0,다사60a36b,"POLYGON ((960300 1936800, 960300 1936900, 9604..."
6,다사604368,11220670,서초구,양재2동,960450.0,1936850.0,0.0,0.0,0.0,22900.000000,0,0,다사60a36b,"POLYGON ((960400 1936800, 960400 1936900, 9605..."
7,다사605368,11220670,서초구,양재2동,960550.0,1936850.0,0.0,0.0,0.0,21700.000000,0,0,다사60b36b,"POLYGON ((960500 1936800, 960500 1936900, 9606..."
8,다사606368,11220670,서초구,양재2동,960650.0,1936850.0,0.0,0.0,0.0,21500.000000,0,0,다사60b36b,"POLYGON ((960600 1936800, 960600 1936900, 9607..."
9,다사607368,11220670,서초구,양재2동,960750.0,1936850.0,0.0,0.0,0.0,30233.336667,0,0,다사60b36b,"POLYGON ((960700 1936800, 960700 1936900, 9608..."


,ADM_CD,시군구,행정동,geometry
0,11010530,종로구,사직동,"POLYGON ((197702.069 553187.311, 197703.431 55..."
1,11010540,종로구,삼청동,"POLYGON ((197980.839 555346.068, 197995.421 55..."
2,11010550,종로구,부암동,"POLYGON ((196621.023 556395.88, 196628.324 556..."
3,11010560,종로구,평창동,"POLYGON ((197800.72 559064.245, 197782.581 558..."
4,11010570,종로구,무악동,"POLYGON ((196444.745 553384.564, 196471.618 55..."
5,11010580,종로구,교남동,"POLYGON ((196720.241 553105.144, 196721.382 55..."
6,11010600,종로구,가회동,"POLYGON ((199036.605 554473.75, 199030.218 554..."
7,11010610,종로구,종로1·2·3·4가동,"POLYGON ((199061.502 554230.746, 199069.577 55..."
8,11010630,종로구,종로5·6가동,"POLYGON ((200757.039 553017.729, 200756.967 55..."
9,11010640,종로구,이화동,"POLYGON ((200510.908 553979.009, 200516.753 55..."


In [ ]:
AGE_GENDER_PATH = GRID_PATH / "격자100m_성연령별인구_2024_10"

def read_shp_file(age):
    PATH = AGE_GENDER_PATH / age
    
    shp_list = list(PATH.rglob("vl_blk.shp")) # rglob은 하위 폴더까지 모두 탐색
    
    print(f'{age} shp 데이터 개수: {len(shp_list)}')
    
    gdf_list = []
    
    for shp in shp_list:
        temp = gpd.read_file(shp,
                             encoding = 'utf-8')
        temp["연령대"] = shp.parent.parent.parent.name
        temp["성별"] = shp.parent.parent.name
        temp["시군구"] = shp.parent.name
        
        gdf_list.append(temp)
    
    gdf_con = pd.concat(gdf_list, 
                       ignore_index= True,
                       axis = 0) # 데이터 파일마다 중복되는 인덱스 무시하고 새로 생성
    
    result = gpd.GeoDataFrame(gdf_con,
                              geometry = "geometry",
                              crs = gdf_list[0].crs)
    
    print(f'\ncrs: {result.crs}')
    print(f'\n지오타입 {result.geometry.geom_type.unique()}')
    print(f'\n데이터 구조 {result.shape}')
    display(result.head())
        
    return result

category_list = [
 '총인구',
 '유아인구', # 만 0 - 4세
 '유소년인구', # 만 0 -14세
 '초등학생인구',
 '중학생인구',
 '고등학생인구', 
 '20대인구',
 '30대인구',
 '40대인구',
 '50대인구',
 '60대인구',
 '70대인구',
 '80대인구',
 '90대인구',
 '100세이상인구'
 ]


grid_list = []
for age in category_list:
    result = read_shp_file(age)
    
    grid_list.append(result)
    
grid_stat = pd.concat(grid_list,
                      axis = 0,
                      ignore_index=True)
print('='*100)
print(f'\ncrs: {grid_stat.crs}')
print(f'\n지오타입 {grid_stat.geometry.geom_type.unique()}')
print(f'\n데이터 구조 {grid_stat.shape}')
display(grid_stat.head(20))

총인구 shp 데이터 개수: 75

crs: EPSG:5179

지오타입 <ArrowStringArray>
['Polygon']
Length: 1, dtype: str

데이터 구조 (194043, 7)


,gid,lbl,val,geometry,연령대,성별,시군구
0,다사565474,NaN,NaN,"POLYGON ((956500 1947400, 956500 1947500, 9566...",총인구,총인구,강남구
1,다사565475,NaN,NaN,"POLYGON ((956500 1947500, 956500 1947600, 9566...",총인구,총인구,강남구
2,다사566473,NaN,NaN,"POLYGON ((956600 1947300, 956600 1947400, 9567...",총인구,총인구,강남구
3,다사566474,NaN,NaN,"POLYGON ((956600 1947400, 956600 1947500, 9567...",총인구,총인구,강남구
4,다사566475,NaN,NaN,"POLYGON ((956600 1947500, 956600 1947600, 9567...",총인구,총인구,강남구
5,다사566476,NaN,NaN,"POLYGON ((956600 1947600, 956600 1947700, 9567...",총인구,총인구,강남구
6,다사567472,NaN,NaN,"POLYGON ((956700 1947200, 956700 1947300, 9568...",총인구,총인구,강남구
7,다사567473,NaN,NaN,"POLYGON ((956700 1947300, 956700 1947400, 9568...",총인구,총인구,강남구
8,다사567474,NaN,NaN,"POLYGON ((956700 1947400, 956700 1947500, 9568...",총인구,총인구,강남구
9,다사567475,NaN,NaN,"POLYGON ((956700 1947500, 956700 1947600, 9568...",총인구,총인구,강남구


유아인구 shp 데이터 개수: 75

crs: EPSG:5179

지오타입 <ArrowStringArray>
['Polygon']
Length: 1, dtype: str

데이터 구조 (194043, 7)


,gid,lbl,val,geometry,연령대,성별,시군구
0,다사565474,NaN,NaN,"POLYGON ((956500 1947400, 956500 1947500, 9566...",유아인구,총인구,강남구
1,다사565475,NaN,NaN,"POLYGON ((956500 1947500, 956500 1947600, 9566...",유아인구,총인구,강남구
2,다사566473,NaN,NaN,"POLYGON ((956600 1947300, 956600 1947400, 9567...",유아인구,총인구,강남구
3,다사566474,NaN,NaN,"POLYGON ((956600 1947400, 956600 1947500, 9567...",유아인구,총인구,강남구
4,다사566475,NaN,NaN,"POLYGON ((956600 1947500, 956600 1947600, 9567...",유아인구,총인구,강남구
5,다사566476,NaN,NaN,"POLYGON ((956600 1947600, 956600 1947700, 9567...",유아인구,총인구,강남구
6,다사567472,NaN,NaN,"POLYGON ((956700 1947200, 956700 1947300, 9568...",유아인구,총인구,강남구
7,다사567473,NaN,NaN,"POLYGON ((956700 1947300, 956700 1947400, 9568...",유아인구,총인구,강남구
8,다사567474,NaN,NaN,"POLYGON ((956700 1947400, 956700 1947500, 9568...",유아인구,총인구,강남구
9,다사567475,NaN,NaN,"POLYGON ((956700 1947500, 956700 1947600, 9568...",유아인구,총인구,강남구


유소년인구 shp 데이터 개수: 75

crs: EPSG:5179

지오타입 <ArrowStringArray>
['Polygon']
Length: 1, dtype: str

데이터 구조 (194043, 7)


...

,gid,lbl,val,geometry,연령대,성별,시군구
0,다사565474,NaN,NaN,"POLYGON ((956500 1947400, 956500 1947500, 9566...",90대인구,총인구,강남구
1,다사565475,NaN,NaN,"POLYGON ((956500 1947500, 956500 1947600, 9566...",90대인구,총인구,강남구
2,다사566473,NaN,NaN,"POLYGON ((956600 1947300, 956600 1947400, 9567...",90대인구,총인구,강남구
3,다사566474,NaN,NaN,"POLYGON ((956600 1947400, 956600 1947500, 9567...",90대인구,총인구,강남구
4,다사566475,NaN,NaN,"POLYGON ((956600 1947500, 956600 1947600, 9567...",90대인구,총인구,강남구
5,다사566476,NaN,NaN,"POLYGON ((956600 1947600, 956600 1947700, 9567...",90대인구,총인구,강남구
6,다사567472,NaN,NaN,"POLYGON ((956700 1947200, 956700 1947300, 9568...",90대인구,총인구,강남구
7,다사567473,NaN,NaN,"POLYGON ((956700 1947300, 956700 1947400, 9568...",90대인구,총인구,강남구
8,다사567474,NaN,NaN,"POLYGON ((956700 1947400, 956700 1947500, 9568...",90대인구,총인구,강남구
9,다사567475,NaN,NaN,"POLYGON ((956700 1947500, 956700 1947600, 9568...",90대인구,총인구,강남구


100세이상인구 shp 데이터 개수: 75

crs: EPSG:5179

지오타입 <ArrowStringArray>
['Polygon']
Length: 1, dtype: str

데이터 구조 (194043, 7)


,gid,lbl,val,geometry,연령대,성별,시군구
0,다사565474,NaN,NaN,"POLYGON ((956500 1947400, 956500 1947500, 9566...",100세이상인구,총인구,강남구
1,다사565475,NaN,NaN,"POLYGON ((956500 1947500, 956500 1947600, 9566...",100세이상인구,총인구,강남구
2,다사566473,NaN,NaN,"POLYGON ((956600 1947300, 956600 1947400, 9567...",100세이상인구,총인구,강남구
3,다사566474,NaN,NaN,"POLYGON ((956600 1947400, 956600 1947500, 9567...",100세이상인구,총인구,강남구
4,다사566475,NaN,NaN,"POLYGON ((956600 1947500, 956600 1947600, 9567...",100세이상인구,총인구,강남구
5,다사566476,NaN,NaN,"POLYGON ((956600 1947600, 956600 1947700, 9567...",100세이상인구,총인구,강남구
6,다사567472,NaN,NaN,"POLYGON ((956700 1947200, 956700 1947300, 9568...",100세이상인구,총인구,강남구
7,다사567473,NaN,NaN,"POLYGON ((956700 1947300, 956700 1947400, 9568...",100세이상인구,총인구,강남구
8,다사567474,NaN,NaN,"POLYGON ((956700 1947400, 956700 1947500, 9568...",100세이상인구,총인구,강남구
9,다사567475,NaN,NaN,"POLYGON ((956700 1947500, 956700 1947600, 9568...",100세이상인구,총인구,강남구



crs: EPSG:5179

지오타입 <ArrowStringArray>
['Polygon']
Length: 1, dtype: str

데이터 구조 (2910645, 7)


,gid,lbl,val,geometry,연령대,성별,시군구
0,다사565474,NaN,NaN,"POLYGON ((956500 1947400, 956500 1947500, 9566...",총인구,총인구,강남구
1,다사565475,NaN,NaN,"POLYGON ((956500 1947500, 956500 1947600, 9566...",총인구,총인구,강남구
2,다사566473,NaN,NaN,"POLYGON ((956600 1947300, 956600 1947400, 9567...",총인구,총인구,강남구
3,다사566474,NaN,NaN,"POLYGON ((956600 1947400, 956600 1947500, 9567...",총인구,총인구,강남구
4,다사566475,NaN,NaN,"POLYGON ((956600 1947500, 956600 1947600, 9567...",총인구,총인구,강남구
5,다사566476,NaN,NaN,"POLYGON ((956600 1947600, 956600 1947700, 9567...",총인구,총인구,강남구
6,다사567472,NaN,NaN,"POLYGON ((956700 1947200, 956700 1947300, 9568...",총인구,총인구,강남구
7,다사567473,NaN,NaN,"POLYGON ((956700 1947300, 956700 1947400, 9568...",총인구,총인구,강남구
8,다사567474,NaN,NaN,"POLYGON ((956700 1947400, 956700 1947500, 9568...",총인구,총인구,강남구
9,다사567475,NaN,NaN,"POLYGON ((956700 1947500, 956700 1947600, 9568...",총인구,총인구,강남구


## 격자 통계 데이터 전처리 및 파생변수(6-14세, 15-19세) 생성
    # 결측치 처리
        # 인구수 결측치는 해당 성별·연령대 인구가 없거나 개인정보보호 기준에 따라 값이 제공되지 않은 경우로 판단
        # 이전 격자 인구 처리 방식과 동일하게 0으로 처리 후 이후 보정 단계에서 총량 일치 여부를 점검

    # 파생변수 생성
        # 6-14세: 유소년 인구(0-14세) - 유아 인구(0-5세)
        # 14-19세: 0-19세(총 인구 - 20세 이상)

In [53]:
# 칼럼 정리
grid_stat_cleaning = grid_stat[["gid", "시군구", 'val', '성별', '연령대', 'geometry']].copy()

# 칼럼명 변경
grid_stat_cleaning = grid_stat_cleaning.rename(columns = {"gid": "GRID_CD",
                                                          "val": "인구수"})

# 데이터 타입 변경
grid_stat_cleaning["인구수"]=  pd.to_numeric(grid_stat_cleaning["인구수"],
                                          errors = "coerce")
print(grid_stat_cleaning['인구수'].dtypes)

# 결측치 처리
grid_stat_clean = grid_stat_cleaning.copy()
grid_stat_clean["인구수"] = grid_stat_clean["인구수"].fillna(0)

print('전처리 확인')
print(grid_stat_clean['인구수'].isna().sum())

float64
전처리 확인
0


In [56]:
# 데이터 구조 정리: 그룹화
grid_stat_group = grid_stat_clean.groupby(["GRID_CD", "시군구", "성별", "연령대"], as_index=False)["인구수"].sum()

print(f"집계 전 구조: {grid_stat_clean.shape}")
print(f"집계 후 구조: {grid_stat_group.shape}")
print(f"중복 확인: {grid_stat_group[['GRID_CD', '시군구', '성별', '연령대']].duplicated().sum()}")

집계 전 구조: (2910645, 6)
집계 후 구조: (2910645, 5)
중복 확인: 0


In [68]:
# 파생 연령대 생성(6-14, 15-19세) 위해 wide 형태로 변경
grid_stat_wide = grid_stat_group.pivot_table(index=["GRID_CD", "시군구","성별"],
                                            columns = "연령대",
                                            values = "인구수",
                                            aggfunc = "sum",
                                            fill_value = 0
).reset_index()

grid_stat_wide.columns.name = None

age_over_20_cols = ["20대인구", "30대인구", "40대인구", "50대인구",
    "60대인구", "70대인구", "80대인구", "90대인구", "100세이상인구"]

grid_stat_wide["0-19세"] = grid_stat_wide["총인구"] - grid_stat_wide[age_over_20_cols].sum(axis=1)

# 유아 인구 (0-5세)
grid_stat_wide["0-5세"] = grid_stat_wide["유아인구"]

# 6-14세
grid_stat_wide["6-14세"] = grid_stat_wide["유소년인구"] - grid_stat_wide["0-5세"]

# 15-19세
grid_stat_wide["15-19세"] = grid_stat_wide["0-19세"] - grid_stat_wide["유소년인구"]

print("0-19세 음수:", (grid_stat_wide["0-19세"] < 0 ).sum())
print("6~14세 음수:", (grid_stat_wide["6-14세"] < 0).sum())
print("15~19세 음수:", (grid_stat_wide["15-19세"] < 0).sum())

0-19세 음수: 0
6~14세 음수: 0
15~19세 음수: 0


In [ ]:
# 데이터 점검: 연령별 총인구와 남여합 비교
age_cols = [
    "0-5세", "6-14세", "15-19세",
    "20대인구", "30대인구", "40대인구", "50대인구",
    "60대인구", "70대인구", "80대인구", "90대인구", "100세이상인구"
]
stat_total = grid_stat_wide[grid_stat_wide["성별"] == '총인구'].set_index("GRID_CD")[age_cols]
stat_male = grid_stat_wide[grid_stat_wide["성별"] == '남성'].set_index("GRID_CD")[age_cols]
stat_female = grid_stat_wide[grid_stat_wide["성별"] == '여성'].set_index("GRID_CD")[age_cols]

stat_whole = stat_total - (stat_female + stat_male)

display(stat_whole.describe())

# 개인정보 보호, 마스킹, 반올림 등의 영향으로 인해 총인구수와 남여 인구수 합이 가법적으로 일치하지 않도록 설계된 것으로 보임
# 파생변수(6-14세, 15-19세)에서는 평균적으로 남여합이 많았으며, 나머지 연령대에서는 총인구수가 남여 합을 상회하는 수치
# 격자별 인구 통계 테이블에서는 앞선 분석에서 추정한 문화누리대상자에 대한 남여 비율 기반 추정이 핵심이므로,
# 총 인구수는 검증 혹은 보조 데이터로 활용할 필요가 있다고 판단, 별도의 처리를 하지 않기로 결정.

,0-5세,6-14세,15-19세,20대인구,30대인구,40대인구,50대인구,60대인구,70대인구,80대인구,90대인구,100세이상인구
count,64681.000000,64681.000000,64681.000000,64681.000000,64681.000000,64681.000000,64681.000000,64681.000000,64681.000000,64681.000000,64681.000000,64681.000000
mean,0.631468,-0.122633,-3.150755,0.289513,0.253877,0.278567,0.254603,0.291925,0.413104,0.764614,0.188834,0.000216
std,1.898720,2.479069,6.495554,1.284876,1.211227,1.283601,1.231487,1.324790,1.554523,2.010806,1.056253,0.039320
min,0.000000,-10.000000,-53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,-4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,8.000000


## 문화누리대상자 성별·연령대별 인구수 산출
#### 추정방법: 격자 내 연령성별 인구 비중(격자 내 연령별 성별 인구수 / 격자별 전연령 인구수 합) * 격자 별 문화누리 추정 인구수

인구 비중 데이터 검토
- 인구 비중 합이 1인 격자 개수 28878
- 인구 비중 합이 0인 격자 개수 35590
- 인구 비중 합이 1이 아니면서 0도 아닌 격자 개수 213
- 이상치 후보 격자 확인 결과, 실제 인구비중합은 1로, 부동소수점으로 인한 집계 오류였던 것으로 확인되었다.

이상치 후보 점검
- 격자별 문화누리대상자 추정 인구수와 성별·연령대 인구비중의 관계를 점검함.
- 문화누리대상자 추정 인구수가 0이나 성별·연령대 인구비중합이 1인 격자는 일반 인구는 존재하지만 문화누리대상자 추정값이 0인 격자로 판단함.
- 문화누리대상자 추정 인구수가 0보다 크나 성별·연령대 인구비중합이 0인 격자는 성별·연령대 배분 기준이 없는 격자로 판단함.

행정동 비중 대체
- 성별·연령대 인구비중합이 0이면서 문화누리대상자 추정 인구수가 존재하는 격자는 그대로 배분할 경우 문화누리대상자 총량이 소실됨.
- 해당 격자는 같은 시군구·행정동의 성별·연령대 인구비중으로 대체함.
- 격자 단위 성별·연령대 인구비중이 존재하는 경우에는 기존 격자 비중을 우선 적용함.

문화누리대상자 성별·연령대별 배분
- 최종 인구비중 = 격자 성별·연령대 비중 우선 적용, 비중이 없는 경우 행정동 성별·연령대 비중 적용.
- 문화누리대상자 성별·연령대별 인구수 = 격자별 문화누리대상자 추정 인구수 × 최종 인구비중.
- 배분 후 격자별 문화누리대상자 총량이 보존되는지 검토함.
- 총량 오차는 부동소수점 수준으로만 나타났으며, np.isclose 기준 오차가 있는 격자는 0개로 확인됨.

정수화 처리
- 문화누리대상자 성별·연령대별 인구수는 사람 수이므로 최종적으로 정수화함.
- 단순 반올림은 격자별 총량을 깨뜨릴 수 있으므로, 내림값을 먼저 계산한 뒤 소수점 잔여가 큰 성별·연령대 순서대로 잔여 인원을 1명씩 배분함.
- 정수화 목표 총량은 격자별 문화누리대상자 추정 인구수의 반올림값으로 설정함.
- 정수화 후 성별·연령대별 인구수 합계가 격자별 목표 정수 총량과 일치하는지 검토함.


In [ ]:
est_cols = ["GRID_CD", "시군구", "성별",
    "0-5세", "6-14세", "15-19세",
    "20대인구", "30대인구", "40대인구", "50대인구",
    "60대인구", "70대인구", "80대인구", "90대인구", "100세이상인구"
]

grid_est = grid_stat_wide[est_cols].copy()
grid_est = grid_est[grid_est["성별"].isin(['남성', '여성'])].copy()

# wide -> long
grid_age_gender = grid_est.melt(
    id_vars=["GRID_CD", "시군구", "성별"],
    value_vars=age_cols,
    var_name="연령대",
    value_name="인구수"
)

print('데이터 구조:', grid_age_gender.shape)
# 데이터 구조: (1552344, 5)

# 기본 격자 테이블에서 중복 시군구의 격자들을 격자 코드 중심으로 인구를 합산 한 뒤, 중심점 기준으로 병합, 하나의 시군구에 귀속시켰다.
# 인구 통계 격자 데이터에서도 동일한 문제가 반복될 수 있어, 격자코드 - 성별 - 연령대 중복 값을 확인 후 인구 수를 하나로 통합하였다.
 
grid_gu_count = (
    grid_age_gender
    .groupby("GRID_CD")["시군구"]
    .nunique()
    .reset_index(name="시군구_개수")
)

print("복수 시군구에 포함된 GRID_CD 수:")
print((grid_gu_count["시군구_개수"] > 1).sum())

# 최소 인구수 0으로 처리
grid_age_gender_clean["인구수"] = grid_age_gender_clean["인구수"].clip(lower=0)

grid_age_gender_clean = grid_age_gender.groupby(by = ["GRID_CD", "성별", "연령대"], as_index=False).agg({"인구수": 'sum'})

print('\n시군구 병합 후 중복 확인')
print(grid_age_gender_clean[["GRID_CD", "성별", "연령대"]].duplicated().sum())

# 격자 별 남여 인구수 합 칼럼
grid_age_gender_clean["격자별전연령인구수합"] = grid_age_gender_clean.groupby(["GRID_CD"])["인구수"].transform('sum')
grid_age_gender_clean.head()

# 성연령별 인구 비중 칼럼 : 칼럼
grid_age_gender_clean["격자별인구비중"] = np.where(grid_age_gender_clean["격자별전연령인구수합"] > 0,
                                      grid_age_gender_clean["인구수"] / grid_age_gender_clean["격자별전연령인구수합"],
                                      0)

print('\n오류 점검')
print(((grid_age_gender_clean["격자별전연령인구수합"] <= 0) & (grid_age_gender_clean["격자별인구비중"]>0)).sum())


ratio_check = grid_age_gender_clean.groupby("GRID_CD", as_index=False)["격자별인구비중"].sum().rename(columns = {'격자별인구비중': "격자별인구비중합"})
print('\n격자별인구비중 데이터 검토')
print(ratio_check["격자별인구비중합"].describe())
print('인구 비중 합이 1인 격자 개수', (ratio_check["격자별인구비중합"] == 1).sum())
print('인구 비중 합이 0인 격자 개수', (ratio_check["격자별인구비중합"] == 0).sum())
print('인구 비중 합이 1이 아니면서 0도 아닌 격자 개수', ((ratio_check["격자별인구비중합"] != 1) & (ratio_check["격자별인구비중합"] != 0)).sum())

ratio_check[(ratio_check["격자별인구비중합"] != 1) & (ratio_check["격자별인구비중합"] != 0)].sort_values('격자별인구비중합', 
                                                                                         ascending=True)

# 인구 비중 합이 1인 격자 개수 28878
# 인구 비중 합이 0인 격자 개수 35590
# 인구 비중 합이 1이 아니면서 0도 아닌 격자 개수 213
# 이상치 후보 격자 확인 결과, 실제 인구비중합은 1로, 부동소수점으로 인한 집계 오류였던 것으로 확인되었다.


데이터 구조: (1552344, 5)
복수 시군구에 포함된 GRID_CD 수:
2984

시군구 병합 후 중복 확인
0

오류 점검
0

격자별인구비중 데이터 검토
count    61652.000000
mean         0.468955
std          0.499039
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
Name: 격자별인구비중합, dtype: float64
인구 비중 합이 1인 격자 개수 28694
인구 비중 합이 0인 격자 개수 32740
인구 비중 합이 1이 아니면서 0도 아닌 격자 개수 218


,GRID_CD,격자별인구비중합
1164,다사384533,1.0
1556,다사390516,1.0
1973,다사396444,1.0
2069,다사397506,1.0
2305,다사400438,1.0
...,...,...
60257,다사689431,1.0
60301,다사689495,1.0
60332,다사690432,1.0
19748,다사509510,1.0


In [ ]:
grid_ratio = grid_age_gender_clean.copy()
grid_ratio.head()

print(grid_ratio[["GRID_CD", "연령대", "성별"]].duplicated().sum())
# 72696

grid_merge = grid.merge(grid_ratio, 
                        on = "GRID_CD",
                        how = 'left')

print('병합 후 점검')
print('결합 전 기본 격자 구조', grid.shape)
print('결합 전 성연령 격자 구조', grid_ratio.shape)
print('줄어든 격자 개수', len(grid_ratio) - len(grid_merge))
print('데이터 구조', grid_merge.shape)
print('결측치', grid_merge.isna().sum())
print('격자_연령_성별 중복', grid_merge[["GRID_CD", "연령대", "성별"]].duplicated().sum())


    # 0
    # 병합 후 점검
    # 결합 전 기본 격자 구조 (60528, 14)
    # 결합 전 성연령 격자 구조 (1479648, 6)
    # 줄어든 격자 개수 26976
    # 데이터 구조 (1452672, 19)
    # 결측치 GRID_CD           0
    # 행정동코드             0
    # 시군구               0
    # 행정동               0
    # 중심점_x             0
    # 중심점_y             0
    # 원본_인구수            0
    # 주거면적              0
    # 주택수               0
    # 공시지가              0
    # 추정_인구수            0
    # 문화누리대상자_추정_인구수    0
    # GRID_CD_500       0
    # geometry          0
    # 성별                0
    # 연령대               0
    # 인구수               0
    # 격자별전연령인구수합        0
    # 격자별인구비중           0
    # dtype: int64
    # 격자_연령_성별 중복 0


0
병합 후 점검
결합 전 기본 격자 구조 (60528, 14)
결합 전 성연령 격자 구조 (1479648, 6)
줄어든 격자 개수 26976
데이터 구조 (1452672, 19)
결측치 GRID_CD           0
행정동코드             0
시군구               0
행정동               0
중심점_x             0
중심점_y             0
원본_인구수            0
주거면적              0
주택수               0
공시지가              0
추정_인구수            0
문화누리대상자_추정_인구수    0
GRID_CD_500       0
geometry          0
성별                0
연령대               0
인구수               0
격자별전연령인구수합        0
격자별인구비중           0
dtype: int64
격자_연령_성별 중복 0


In [149]:
# 칼럼명 변경
age_name_map = {
    "20대인구": "20-29세",
    "30대인구": "30-39세",
    "40대인구": "40-49세",
    "50대인구": "50-59세",
    "60대인구": "60-69세",
    "70대인구": "70-79세",
    "80대인구": "80-89세",
    "90대인구": "90-99세",
    "100세이상인구": "100세-"
}

grid_merge["연령대"] = grid_merge["연령대"].replace(age_name_map)


In [ ]:
# 성 연령별 추정 인구수

# 이상치 후보 검토
grid_out = grid_merge.groupby("GRID_CD", as_index = False).agg({"문화누리대상자_추정_인구수": 'first',
                                              "격자별인구비중": 'sum'}
                                             ).rename(columns = {"격자별인구비중": "격자별인구비중합"})

over_idx = (grid_out["문화누리대상자_추정_인구수"] == 0) & (grid_out["격자별인구비중합"] != 0)
under_idx = (grid_out["문화누리대상자_추정_인구수"] != 0) & (grid_out["격자별인구비중합"] == 0)

print(over_idx.sum())
display(grid_out[over_idx].sort_values('격자별인구비중합', ascending=False))

print(under_idx.sum())
display(grid_out[under_idx].sort_values('문화누리대상자_추정_인구수', ascending=False))


# 일부 격자에서는 문화누리대상자 추정 인구수는 존재하지만 성별·연령별 격자 인구비중이 0으로 나타났다. 
# 해당 격자에 격자 단위 성별·연령 분포를 적용하면 문화누리대상자 총량이 소실되므로, 같은 시군구·행정동의 성별·연령별 인구비중으로 대체하였다. \
# 이를 통해 격자별 문화누리대상자 총량을 보존하면서 성별·연령대별 배분이 가능하도록 하였다.

690


,GRID_CD,문화누리대상자_추정_인구수,격자별인구비중합
60249,다사710502,0,1.0
3688,다사413509,0,1.0
58335,다사679462,0,1.0
58334,다사679461,0,1.0
58235,다사678467,0,1.0
...,...,...,...
32483,다사568446,0,1.0
28870,다사553514,0,1.0
27659,다사548435,0,1.0
10497,다사470466,0,1.0


209


,GRID_CD,문화누리대상자_추정_인구수,격자별인구비중합
30390,다사559604,3,0.0
53857,다사646428,3,0.0
36719,다사583642,3,0.0
31647,다사564636,3,0.0
6415,다사439513,2,0.0
...,...,...,...
58524,다사681441,1,0.0
59316,다사690448,1,0.0
60052,다사704529,1,0.0
60305,다사711520,1,0.0


In [162]:
# 1. 행정동별 성별·연령대 비중 만들기
dong_ratio = (
    grid_merge[grid_merge["격자별전연령인구수합"] > 0]
    .groupby(["시군구", "행정동", "성별", "연령대"], as_index=False)["인구수"]
    .sum()
)

dong_ratio["행정동_전연령인구수합"] = (
    dong_ratio
    .groupby(["시군구", "행정동"])["인구수"]
    .transform("sum")
)

dong_ratio["행정동_인구비중"] = np.where(
    dong_ratio["행정동_전연령인구수합"] > 0,
    dong_ratio["인구수"] / dong_ratio["행정동_전연령인구수합"],
    0
)

dong_ratio = dong_ratio[[
    "시군구", "행정동", "성별", "연령대", "행정동_인구비중"
]]

grid_merge = grid_merge.merge(
    dong_ratio,
    on=["시군구", "행정동", "성별", "연령대"],
    how="left"
)

grid_merge["최종인구비중"] = np.where(
    grid_merge["격자별전연령인구수합"] > 0,
    grid_merge["격자별인구비중"],
    grid_merge["행정동_인구비중"]
)

grid_merge["최종인구비중"] = grid_merge["최종인구비중"].fillna(0)

grid_merge["문화누리대상자_인구수"] = (
    grid_merge["문화누리대상자_추정_인구수"]
    * grid_merge["최종인구비중"]
)

mnc_check = (
    grid_merge
    .groupby("GRID_CD", as_index=False)
    .agg({
        "문화누리대상자_추정_인구수": "first",
        "문화누리대상자_인구수": "sum"
    })
)

mnc_check["오차"] = (
    mnc_check["문화누리대상자_추정_인구수"]
    - mnc_check["문화누리대상자_인구수"]
)

display(mnc_check["오차"].describe())
print("총량 오차 있는 격자 수:", (~np.isclose(mnc_check["오차"], 0)).sum())

count    6.052800e+04
mean     4.372805e-18
std      2.526044e-16
min     -7.105427e-15
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.421085e-14
Name: 오차, dtype: float64

총량 오차 있는 격자 수: 0


In [171]:
grid_merge.head()
grid_merge.columns

Index(['GRID_CD', '행정동코드', '시군구', '행정동', '중심점_x', '중심점_y', '원본_인구수', '주거면적',
       '주택수', '공시지가', '추정_인구수', '문화누리대상자_추정_인구수', 'GRID_CD_500', 'geometry',
       '성별', '연령대', '인구수', '격자별전연령인구수합', '격자별인구비중', '행정동_인구비중', '최종인구비중',
       '문화누리대상자_인구수', '문화누리대상자_인구수_raw', '문화누리대상자_인구수_floor', '문화누리대상자_인구수_소수',
       '추가배분수', '추가배분순위', '추가배분', '문화누리대상자_인구수_정수'],
      dtype='str')

In [169]:
# 정수화 대상 값
grid_merge["문화누리대상자_인구수_raw"] = grid_merge["문화누리대상자_인구수"]

# 내림값과 소수점
grid_merge["문화누리대상자_인구수_floor"] = np.floor(
    grid_merge["문화누리대상자_인구수_raw"]
).astype(int)

grid_merge["문화누리대상자_인구수_소수"] = (
    grid_merge["문화누리대상자_인구수_raw"]
    - grid_merge["문화누리대상자_인구수_floor"]
)

# 격자별 목표 총량
grid_target = (
    grid_merge
    .groupby("GRID_CD", as_index=False)["문화누리대상자_추정_인구수"]
    .first()
)

grid_target["목표정수"] = grid_target["문화누리대상자_추정_인구수"].round().astype(int)

# 격자별 floor 합계
grid_floor_sum = (
    grid_merge
    .groupby("GRID_CD", as_index=False)["문화누리대상자_인구수_floor"]
    .sum()
    .rename(columns={"문화누리대상자_인구수_floor": "floor합계"})
)

grid_target = grid_target.merge(
    grid_floor_sum,
    on="GRID_CD",
    how="left"
)

grid_target["추가배분수"] = (
    grid_target["목표정수"] - grid_target["floor합계"]
).astype(int)

grid_merge = grid_merge.merge(
    grid_target[["GRID_CD", "추가배분수"]],
    on="GRID_CD",
    how="left"
)

grid_merge["추가배분순위"] = (
    grid_merge
    .sort_values(
        ["GRID_CD", "문화누리대상자_인구수_소수"],
        ascending=[True, False]
    )
    .groupby("GRID_CD")
    .cumcount() + 1
)

grid_merge["추가배분"] = np.where(
    grid_merge["추가배분순위"] <= grid_merge["추가배분수"],
    1,
    0
)

grid_merge["문화누리대상자_인구수_정수"] = (
    grid_merge["문화누리대상자_인구수_floor"]
    + grid_merge["추가배분"]
)

int_check = (
    grid_merge
    .groupby("GRID_CD", as_index=False)
    .agg({
        "문화누리대상자_추정_인구수": "first",
        "문화누리대상자_인구수_정수": "sum"
    })
)

int_check["목표정수"] = int_check["문화누리대상자_추정_인구수"].round().astype(int)

int_check["정수화오차"] = (
    int_check["목표정수"]
    - int_check["문화누리대상자_인구수_정수"]
)

display(int_check["정수화오차"].describe())
print("정수화 오차 있는 격자 수:", (int_check["정수화오차"] != 0).sum())

count    60528.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: 정수화오차, dtype: float64

정수화 오차 있는 격자 수: 0


In [176]:
grid_mnc_age_gender_final = grid_merge[[
    "GRID_CD",
    "시군구",
    "행정동",
    "성별",
    "연령대",
    "문화누리대상자_인구수_정수"
]].copy()

grid_mnc_age_gender_final.rename(columns = {"문화누리대상자_인구수_정수": "문화누리대상자_성연령별_인구수"})



print(grid_mnc_age_gender_final.shape)
print(grid_mnc_age_gender_final.isna().sum())
print(grid_mnc_age_gender_final[["GRID_CD", "성별", "연령대"]].duplicated().sum())

display(grid_mnc_age_gender_final.head(30))

(1452672, 6)
GRID_CD           0
시군구               0
행정동               0
성별                0
연령대               0
문화누리대상자_인구수_정수    0
dtype: int64
0


,GRID_CD,시군구,행정동,성별,연령대,문화누리대상자_인구수_정수
0,다사603367,서초구,양재2동,남성,0-5세,0
1,다사603367,서초구,양재2동,남성,100세-,0
2,다사603367,서초구,양재2동,남성,15-19세,0
3,다사603367,서초구,양재2동,남성,20-29세,0
4,다사603367,서초구,양재2동,남성,30-39세,0
5,다사603367,서초구,양재2동,남성,40-49세,0
6,다사603367,서초구,양재2동,남성,50-59세,0
7,다사603367,서초구,양재2동,남성,6-14세,0
8,다사603367,서초구,양재2동,남성,60-69세,0
9,다사603367,서초구,양재2동,남성,70-79세,0


## 격자별 소득 분위 추정
    # 주택가격 혹은 공시지가 사용
        # 일단 미정
        # 차상위계층과 기초생활수급자의 자격 요건 중 하나인 중위소득 50% 이하를 반영할 수 있음
        # 중위소득 50%을 전 격자 기본값으로 설정 후 전체 격자에 대한 공시지가의 분위수를 가중치로 격자별 소득 수준 추정 가능
        # 하지만 차상위·기초생활수급자의 소득 수준과 거주지 환경이 비례한다고 가정하는 것은 논리적 비약일 수 있어 추정 위험이 높음 


    # 가구별 소득수준 기준 중위값
    연도	1인가구	2인가구	3인가구	4인가구	5인가구	6인가구
    2024	1,114,222	1,841,305	2,357,328	2,864,956	3,347,867	3,809,184
    2025	1,196,007	1,966,329	2,512,677	3,048,887	3,554,096	4,032,403

## 데이터 저장

- ANALYSIS TABLE 폴더의 OUTPUT 폴더에 저장
- 파일명: 서울시_격자_100m_문화누리대상자_성연령별_인구수.csv
- 인코딩: utf-8-sig

In [177]:
grid_mnc_age_gender_final.to_csv(OUTPUT_PATH / "서울시_격자_100m_문화누리대상자_성연령별_인구수.csv",
                                index = False,
                                encoding = 'utf-8-sig')